# Milestone 2 - build the WFLW crop cache

**Rebuild note (milestone 6).** The cache now stores more context than the canonical face box, because framing augmentation needs real background to zoom out into rather than zero padding. `preprocess.crop_expand` is 2.2 (context stored) while `preprocess.reference_expand` stays 1.3 (the canonical framing that k = 1.0, the milestone-5 protocol and the Haar calibration all refer to). `cache_size` is 224 so the canonical region stays at about 132 px, comparable to the 128 px of the first cache, which keeps the k = 1.0 numbers comparable across the rebuild.

Attach the WFLW dataset, run all cells (the build decodes all ~10k images once; expect a few minutes), then check:

* **stats**: ~7,500 train + ~2,500 test faces, landmark min/max inside [0,1], round-trip error near 0.00005 px, total about 500 MB at these settings
* **previews**: crops look like centred faces with visible surrounding context, and every point sits on its feature
* more crops will report as padded at the image border than before, which is expected: a 2.2 box around a face near the frame edge runs off the image, and deployment sees the same thing

**Publishing the cache** (so training attaches it instead of raw WFLW):
1. *Save Version* -> *Save & Run All (Commit)*
2. open the committed version -> *Output* tab -> *New Dataset*, or attach the notebook itself as an input
3. training finds it through `cache.dir: auto`


In [ ]:
!rm -rf dms-layer1
!git clone -b claude/facial-landmark-perception-q80yhn https://github.com/keerthanpragnay1728-prog/dms-layer1.git
%cd dms-layer1
# provenance: confirm the commit this run uses BEFORE trusting any number
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!python tests/run_tests.py

In [ ]:
# build both splits -> /kaggle/working/cache (stats + read-back + previews)
!python scripts/build_crop_cache.py --config configs/layer1_base.yaml --split both

In [ ]:
# eyeball check: decoded crops with the cached labels drawn on them
from IPython.display import Image, display
for split in ('train', 'test'):
    print(split)
    display(Image(filename=f'/kaggle/working/cache/{split}_preview.png'))